## Background

In this book, the estimate update form often encountered has this form (2.5):

$$NewEstimate \leftarrow OldEstimate + StepSize[TargetValue - OldEstimate]$$

where $[TargetValue - OldEstimate]$ is the error in the estimate. This estimation
method is more appropriate for nonstationary environments, where the recent
targets have more say on the estimate than the old ones.

### Temporal Difference (TD)

#### Prediction
TD is a combination of  Monte Carlo ideas and dynamic programming (DP) ideas.
Both TD and Monte Carlo (MC) methods use the experience generated by the
policy $\pi$ to learn their estimation $V$ of the true value function
$v_{\pi}$ for state $S_t$. MC methods wait until the end of the episode
to compute the return $G_t$ following the visit of $S_t$ and use it as the
target for  updating $V(S_t)$. For the every-visit MC method, the update looks like this:

$$V(S_t) = V(S_t) + \alpha[G_{t, MC} - V(S_t)]$$

where $G_{t, MC}$ is the actual return at time $t$, and $\alpha$ is a small step.
This follows the update form above, appropriate for non-stationary environments.

TD methods wait one step (TD(0) to be precise) before they can form the target
for the update, using $R_{t+1}$ and $V(S_{t+1})$ as follows:

$$V(S_t) = V(S_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$
where the target here is $  V_{bootstrap}(S_{t}) = R_{t+1} + \gamma V(S_{t+1}) $.
Because TD(0) bases its update in part on an existing estimate, we say that
it is a _bootstrapping_ method.

From the basic definitions we have

$
\begin{align*}
v_{\pi}(S_t = s) &\overset{\cdot}{=}\mathbb{E}[G_t | S_t = s] \\
&= \mathbb{E}[R_{t+1} + \gamma G_{t+1} | S_t = s] \\
&= \mathbb{E}[R_{t+1} + \gamma v_{\pi}(S_{t+1} = s') | S_t = s]
\end{align*}
$

where for the MC method we use an approximation
$G_{t, MC} \approx \mathbb{E}[G_t | S_t = s]$ as the target, whereas for
TD(0) we use $ V_{t, bootstrap}(S_{t} = s) \approx \mathbb{E}[R_{t+1} + \gamma v_{\pi}(S_{t+1} = s') | S_t = s]$
approximation of $ v_{\pi}(S_t = s)$ as the target. For both MC and TD,
the targets are _estimates_ because for MC the true  target
$v_{\pi}(S_t = s) = \mathbb{E}[G_t | S_t = s]$ is not  known,
so the sample return is used instead. Similarly, for the TD case,
$v_{\pi}(S_t = s) = \mathbb{E}[R_{t+1} + \gamma v_{\pi}(S_{t+1} = s') | S_t = s]$
is also unknown, and the sample for the reward $R_t$ and the bootstrap
estimation $V(S_{t+1})$ are used for its approximation instead. TD methods
combine the sampling approach of MC and the bootstrapping of DP.

We refer to TD and Monte Carlo updates as __sample__ updates because
they involve looking ahead to a _sample_ successor state (or state-action pairs)
and rewards, and use the backed-up value to update the old value of the
state or state-action pair. _Sample updates_ differ from the _expected updates_
of DP methods in that they are based on a single sample successor rather than
on a complete distribution of all possible successors.

###### TD Error
In the TD update formulation, the error is known as TD-error and is encountered
frequently in RL, and it is defined as:

$$\delta_{t} \overset{\cdot}{=} R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

If the array $V$ does not change during the episode, as it does not in
for MC, then the MC error can be written as a sum of TD errors:

$$G_t - V(S_t) = \sum_{k=t}^{T - 1}\gamma^{k -t}\delta_{k}$$


#### On-Policy Control, Sarsa
Following the general policy iteration (GPI) process, we need to learn the
state-action value $q_{\pi}$ of the current policy $\pi$ (estimate step),
then improve the policy based on $q_{\pi}$.

For control, learning $q_{\pi}$ for all states $s$ and actions $a$ can be done
in the same way as for $v_{\pi}$:
$$
Q(S_t, A_t) = Q(S_t, A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]
$$

where the update is performed after each observed transition from non-terminal
state $S_t, A_t \rightarrow S_{t+1}, R_{t+1}$, If $S_{t}$ is terminal then
$Q(S_t, A_t) \overset{\cdot}{=} 0$. The update rule uses the experience quintuple
$(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$. This gives rise to the name _Sarsa_
algorithm. For the Sarsa _on-policy_ algorithm, we continuously estimate
the "true" value function $q_{\pi}$ for the behavior policy $\pi$, and improve
the policy by moving towards greediness wrt the current estimate of $q_{\pi}$,
i.e. the GPI process.

###### Convergence
Sarsa converges to an optimal policy $\pi_{*}$  and value function $q_*$ if
GLIE (greedy in the limit with infinite exploration), i.e.:
* State-Action pairs are visited an infinite number of times (coverage)
* Policy converges in the limit to optimal policy
  * e.g., using  a $\varepsilon$-soft policy variant like $\varepsilon$-greedy
  which we incrementally move towards greedy.
  *  Condition for the step-size:
    * $ \sum_{t=1}^{\infty}{\alpha_t} = \infty $
    * $ \sum_{t=1}^{\infty}{\alpha_t ^ 2} < \infty $


#### Off-Policy Control, SarsaMax aka Q-Learning
A frequently used algorithm in RL is SarsaMax or Q-Learning, which uses the
(current) estimate $Q$ of optimal state-action value funciton $q_{*}$ in the
update as follows:

$$
  Q(S_t, A_t) = Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \max_{a}Q(S_{t+1}, a) - Q(S_t, A_t)]
$$

Here, the approximation of $q_{*}$ is directly approximated by using $\max_{a}Q(S_{t+1}, a)$
_independent_ of the policy being followed. Since we use samples
$(S_t, A_t, R_{t+1}, S_{t+1})_{\pi}$ to learn the optimal (target) policy
$\pi_{*}$ which is greedy wrt to $q_{*}$, we call this off-policy.

###### Convergence
Q-Learning is guaranteed to converge to $q_*$ (and relatedly $\pi_*$)
if:
* State-action pairs are visited an infinite number of times (coverage)
* Condition for the step-size:
  * $ \sum_{t=1}^{\infty}{\alpha_t} = \infty $
  * $ \sum_{t=1}^{\infty}{\alpha_t ^ 2} < \infty $

_regardless of how you choose the actions_, unlike Sarsa, where it is
required that we make our behavior policy greedy in the limit (GLIE).

#### On-Policy Control, Expected Sarsa
Expected Sarsa is another variant of the Sarsa type of algorithms, where
for the next step, instead of using the sample $A_{t+1}$ (Sarsa) or
$\max_{a}Q(*, a)$ (Q-Learning) for the next-step value, it evaluates the _expected_
next-step value as follows:

$$
\begin{align*}
  Q(S_t, A_t) &= Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \mathbb{E}_{a \sim \pi(\cdot|S_{t+1})}[Q(S_{t+1}, a)] - Q(S_t, A_t)] \\
              &= Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \sum_{a}\pi(a | S_{t+1})Q(S_{t+1}, a) - Q(S_t, A_t)] \\
              &= Q(S_t, A_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - Q(S_t, A_t)]
\end{align*}
$$

Given the next state, $S_{t+1}$, this algorithm moves _deterministically_ in
the same direction as Sarsa moves in _expectation_ (sample dependent).